# Ekstraksi RWF-2000 → cache .npy (kompatibel dengan cache mahasiswa)

Mereplikasi **persis** resep `extract_frames()` + `VideoDataset` dari `PT-MLR-UFALL.ipynb`
yang menghasilkan cache legacy yang sudah terbukti bagus.

| Aspek | Nilai | Alasan |
|---|---|---|
| Sampling | `skip = frame_count // 16`, seek `CAP_PROP_POS_FRAMES` | sama persis |
| Resize | `cv2.resize(frame, (64, 64))` | sama persis |
| **Urutan kanal** | **BGR (TANPA `cvtColor`)** | cache lama BGR; notebook training memakai `LEGACY_CACHE_COLOR_ORDER="BGR"` |
| Skala | `/255.0` → float32 | sama persis |
| Padding | ulangi frame terakhir | sama persis |
| Label | `['NonViolence', 'Violence']` → 0, 1 | sama persis |
| Output | `RWF_features.npy` (N,16,64,64,3) + `RWF_labels.npy` | nama baru → **aditif** |

> **Sifat aditif.** Notebook ini hanya MEMBUAT file baru. Tidak ada file lama yang dibuka
> untuk ditulis. Ada pengaman `overwrite=False` yang menolak menimpa file yang sudah ada.

## 1. Setup

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import os, gc
from pathlib import Path
import numpy as np
import cv2
from tqdm.auto import tqdm

# ---------- PATH ----------
BV_RAW_DIR = Path("/content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/BV")
CACHE_DIR   = Path("/content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/new_saved_dataset")
DATASET_NAME = "BV"

# ---------- PARAMETER (JANGAN DIUBAH — harus sama dengan cache lama) ----------
SEQUENCE_LENGTH = 16
IMAGE_HEIGHT = IMAGE_WIDTH = 64
CLASSES_LIST = ["NonViolence", "Violence"]   # indeks 0, 1
VIDEO_EXT = {".avi", ".mp4", ".mov", ".mpg", ".mpeg", ".mkv"}

print("BV raw   :", BV_RAW_DIR, "->", "ADA" if BV_RAW_DIR.exists() else "TIDAK ADA")
print("Cache dir :", CACHE_DIR,   "->", "ADA" if CACHE_DIR.exists() else "TIDAK ADA")

BV raw   : /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/BV -> ADA
Cache dir : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/new_saved_dataset -> ADA


## 2. Periksa struktur folder UCF

In [19]:
if BV_RAW_DIR.exists():
    print("Isi", BV_RAW_DIR, ":")
    for p in sorted(BV_RAW_DIR.iterdir()):
        if p.is_dir():
            vids = [f for f in p.iterdir() if f.suffix.lower() in VIDEO_EXT]
            print(f"   [DIR ] {p.name:<20} {len(vids)} video")
        else:
            print(f"   [FILE] {p.name}")

missing = [c for c in CLASSES_LIST if not (BV_RAW_DIR / c).is_dir()]
if missing:
    print(f"\n[!] Subfolder tidak ditemukan: {missing}")
    print("    Struktur yang dibutuhkan: BV/NonViolence/*.avi dan BV/Violence/*.avi")
    print("    Jika nama folder berbeda (mis. 'Fight'/'NonFight'), sesuaikan CLASSES_LIST")
    print("    ATAU buat symlink/rename agar cocok.")
else:
    print("\nStruktur folder OK.")

Isi /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/BV :
   [DIR ] NonViolence          700 video
   [DIR ] Violence             700 video

Struktur folder OK.


## 3. Cek keamanan — pastikan tidak menimpa apa pun

In [20]:
feature_path = CACHE_DIR / f"{DATASET_NAME}_features.npy"
label_path   = CACHE_DIR / f"{DATASET_NAME}_labels.npy"

print("Akan menulis:")
print("  ", feature_path.name, "->", "SUDAH ADA (!)" if feature_path.exists() else "belum ada, aman")
print("  ", label_path.name,   "->", "SUDAH ADA (!)" if label_path.exists()   else "belum ada, aman")

print("\nFile yang SUDAH ADA di cache (tidak akan disentuh):")
for p in sorted(CACHE_DIR.glob("*.npy")):
    print("   ", p.name)

Akan menulis:
   BV_features.npy -> belum ada, aman
   BV_labels.npy -> belum ada, aman

File yang SUDAH ADA di cache (tidak akan disentuh):
    Automatic_features.npy
    Automatic_labels.npy
    HockeyFight_features.npy
    HockeyFight_labels.npy
    MovieFight_features.npy
    MovieFight_labels.npy
    RWF_features.npy
    RWF_labels.npy
    Realife_features.npy
    Realife_labels.npy
    Surveilence_features.npy
    Surveilence_labels.npy
    UCF_features.npy
    UCF_labels.npy
    ViolentFlow_features.npy
    ViolentFlow_labels.npy
    split_indices_trainval.npy
    trainval_features.npy
    trainval_labels.npy


## 4. Fungsi ekstraksi

Disalin verbatim dari `PT-MLR-UFALL.ipynb`. **Jangan diubah** — perbedaan sekecil apa pun
membuat RWF tidak sebanding dengan enam dataset lainnya.

In [21]:
def extract_frames(video_path, sequence_length, image_height, image_width):
    """Replika verbatim dari PT-MLR-UFALL.ipynb. Mengembalikan (T,H,W,3) BGR float32 [0,1]."""
    frames_list = []
    video_reader = cv2.VideoCapture(str(video_path))
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(video_frames_count // sequence_length, 1)

    for frame_counter in range(sequence_length):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break
        resized_frame = cv2.resize(frame, (image_width, image_height))
        normalized_frame = resized_frame / 255.0          # BGR — TANPA cvtColor
        frames_list.append(normalized_frame)

    if len(frames_list) < sequence_length:
        if not frames_list:
            video_reader.release()
            return np.array([], dtype=np.float32)
        frames_list.extend(frames_list[-1:] * (sequence_length - len(frames_list)))

    video_reader.release()
    return np.array(frames_list, dtype=np.float32)


def build_cache(raw_dir, cache_dir, dataset_name, classes_list,
                sequence_length=SEQUENCE_LENGTH,
                image_height=IMAGE_HEIGHT, image_width=IMAGE_WIDTH,
                overwrite=False):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    fpath = cache_dir / f"{dataset_name}_features.npy"
    lpath = cache_dir / f"{dataset_name}_labels.npy"

    if (fpath.exists() or lpath.exists()) and not overwrite:
        raise FileExistsError(
            f"{fpath.name} / {lpath.name} sudah ada. "
            "Set overwrite=True HANYA bila memang ingin menimpa.")

    all_files = []
    for class_name in classes_list:
        class_dir = Path(raw_dir) / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Subfolder kelas tidak ada: {class_dir}")
        vids = sorted(f for f in class_dir.iterdir() if f.suffix.lower() in VIDEO_EXT)
        all_files += [(class_name, f) for f in vids]
        print(f"  {class_name:<14}: {len(vids)} video")

    features, labels, skipped = [], [], []
    for class_name, video_path in tqdm(all_files, desc="Ekstraksi", unit="video"):
        class_index = classes_list.index(class_name)
        frames = extract_frames(video_path, sequence_length, image_height, image_width)
        if len(frames) == sequence_length:
            features.append(frames)
            labels.append(class_index)
        else:
            skipped.append(str(video_path))

    features_np = np.array(features, dtype=np.float32)
    labels_np = np.array(labels)

    np.save(fpath, features_np)
    np.save(lpath, labels_np)

    print(f"\nTersimpan: {fpath.name} {features_np.shape} ({features_np.nbytes/1024**2:.1f} MB)")
    print(f"Tersimpan: {lpath.name} {labels_np.shape}")
    if skipped:
        print(f"[!] {len(skipped)} video dilewati (gagal dibaca). Contoh: {skipped[:5]}")
    return features_np, labels_np, skipped

print("fungsi siap")

fungsi siap


## 5. Uji coba kecil dulu

Jalankan pada beberapa video untuk memastikan pembacaan berhasil sebelum memproses 2.000 klip.

In [22]:
sample_dir = BV_RAW_DIR / "Violence"
samples = sorted(f for f in sample_dir.iterdir() if f.suffix.lower() in VIDEO_EXT)[:3]

for s in samples:
    fr = extract_frames(s, SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH)
    if len(fr) == 0:
        print(f"[GAGAL] {s.name}")
        continue
    m = fr.reshape(-1, 3).mean(axis=0)
    print(f"{s.name[:34]:<36} shape={fr.shape} range=[{fr.min():.3f},{fr.max():.3f}] "
          f"ch={m.round(3)} ch0/ch2={m[0]/(m[2]+1e-9):.3f}")

print("\nch0/ch2 < 1 menandakan BGR (kanal-0 = Blue), konsisten dengan cache lama.")

VIOLENCE_10.mp4                      shape=(16, 64, 64, 3) range=[0.000,0.984] ch=[0.392 0.387 0.391] ch0/ch2=1.000
VIOLENCE_1000.mp4                    shape=(16, 64, 64, 3) range=[0.000,1.000] ch=[0.302 0.316 0.325] ch0/ch2=0.928
VIOLENCE_1004.mp4                    shape=(16, 64, 64, 3) range=[0.000,1.000] ch=[0.324 0.331 0.331] ch0/ch2=0.979

ch0/ch2 < 1 menandakan BGR (kanal-0 = Blue), konsisten dengan cache lama.


## 6. Ekstraksi penuh

Untuk 2.000 klip, perkirakan 15–40 menit tergantung kecepatan I/O Drive.

In [23]:
features_np, labels_np, skipped = build_cache(
    raw_dir=BV_RAW_DIR,
    cache_dir=CACHE_DIR,
    dataset_name=DATASET_NAME,
    classes_list=CLASSES_LIST,
    overwrite=False,          # pengaman — jangan diubah kecuali sengaja menimpa
)

  NonViolence   : 700 video
  Violence      : 700 video


Ekstraksi:   0%|          | 0/1400 [00:00<?, ?video/s]


Tersimpan: BV_features.npy (1400, 16, 64, 64, 3) (1050.0 MB)
Tersimpan: BV_labels.npy (1400,)


## 7. Verifikasi kompatibilitas dengan cache lama

Pastikan RWF konsisten dengan enam dataset lainnya sebelum dipakai.

In [24]:
import pandas as pd

NAMES = {"TrainVal":"trainval", "HF":"HockeyFight", "MF":"MovieFight",
         "VCF":"ViolentFlow", "AVDV":"Automatic", "SCFD":"Surveilence",
         "RLVS":"Realife", "RWF":"RWF", "UCF":"UCF", "BV":"BV"}

rows = []
for tag, cname in NAMES.items():
    ff = CACHE_DIR / f"{cname}_features.npy"
    lf = CACHE_DIR / f"{cname}_labels.npy"
    if not (ff.exists() and lf.exists()):
        rows.append({"dataset": tag, "status": "HILANG"}); continue
    a = np.load(ff, mmap_mode="r"); y = np.load(lf)
    s = np.asarray(a[:min(30, len(a))], dtype=np.float32)
    if s.max() > 1.5: s = s / 255.0
    m = s.reshape(-1, s.shape[-1]).mean(axis=0)
    u, c = np.unique(y, return_counts=True)
    rows.append({
        "dataset": tag, "status": "ok", "n": len(y),
        "shape": tuple(a.shape[1:]), "dtype": str(a.dtype),
        "range": f"[{float(np.asarray(a[:3]).min()):.2f},{float(np.asarray(a[:3]).max()):.2f}]",
        "ch0/ch2": round(float(m[0]/(m[2]+1e-9)), 3),
        "labels": dict(zip(u.tolist(), c.tolist())),
    })

df = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(df.to_string(index=False))

ok = df[df.status == "ok"]
if "BV" in ok.dataset.values and "TrainVal" in ok.dataset.values:
    r = ok.set_index("dataset")["ch0/ch2"]
    d = abs(r["BV"] - r["TrainVal"])
    print(f"\nSelisih ch0/ch2 BV vs TrainVal = {d:.3f}")
    print("  -> KONSISTEN, konvensi warna sama." if d < 0.25
          else "  -> [!] MENYIMPANG, periksa builder cache.")
    shapes = set(tuple(s) for s in ok["shape"])
    print(f"Bentuk per-klip: {shapes} -> " + ("seragam" if len(shapes) == 1 else "[!] TIDAK seragam"))

 dataset status    n           shape   dtype       range  ch0/ch2             labels
TrainVal     ok 3683 (16, 64, 64, 3) float32 [0.00,1.00]    0.958 {0: 1793, 1: 1890}
      HF     ok  100 (16, 64, 64, 3) float32 [0.00,1.00]    0.941     {0: 50, 1: 50}
      MF     ok   20 (16, 64, 64, 3) float32 [0.00,0.82]    0.815     {0: 10, 1: 10}
     VCF     ok   26 (16, 64, 64, 3) float32 [0.00,1.00]    0.915     {0: 13, 1: 13}
    AVDV     ok   35 (16, 64, 64, 3) float32 [0.00,1.00]    0.933     {0: 12, 1: 23}
    SCFD     ok   30 (16, 64, 64, 3) float32 [0.00,1.00]    1.008     {0: 15, 1: 15}
    RLVS     ok  200 (16, 64, 64, 3) float32 [0.00,1.00]    0.912   {0: 100, 1: 100}
     RWF     ok 1998 (16, 64, 64, 3) float32 [0.00,1.00]    0.975  {0: 1000, 1: 998}
     UCF     ok  100 (16, 64, 64, 3) float32 [0.00,1.00]    0.959     {0: 50, 1: 50}
      BV     ok 1400 (16, 64, 64, 3) float32 [0.00,1.00]    0.967   {0: 700, 1: 700}

Selisih ch0/ch2 BV vs TrainVal = 0.009
  -> KONSISTEN, konvensi 

---
## Langkah berikutnya

**RWF sebagai domain held-out murni (direkomendasikan).** Jangan masukkan ke pool latih.
Cukup tambahkan ke `TEST_SPECS` di notebook training/evaluasi:

```python
"RWF": {"raw_dir": DATA_ROOT / "RWF", "cache_name": "RWF"},
```

Lalu evaluasi checkpoint yang **sudah ada** — tanpa retrain. Ini memberi uji generalisasi
lintas domain yang sesungguhnya, dan menghemat ~7,5 jam dibanding memasukkannya ke pool latih.

## Bila nama subfolder berbeda

Distribusi RWF-2000 kadang memakai `Fight` / `NonFight`. Bila demikian, ubah:

```python
CLASSES_LIST = ["NonFight", "Fight"]   # urutan menentukan label 0 dan 1
```

⚠️ **Urutan wajib dipertahankan**: indeks 0 = non-violence, indeks 1 = violence — sama dengan
enam dataset lainnya. Tertukar berarti seluruh evaluasi RWF terbalik.

## Bila RWF punya split train/val bawaan

RWF-2000 resmi terbagi menjadi `train/` (1.600) dan `val/` (400). Bila struktur Anda demikian,
jalankan `build_cache` dua kali dengan `dataset_name="RWF_train"` dan `"RWF_val"`, atau arahkan
`RWF_RAW_DIR` ke subfolder yang diinginkan. Untuk uji held-out, gunakan **seluruh 2.000 klip**
atau bagian `val` saja — sebutkan pilihan itu secara eksplisit di paper.